In [ ]:
import os
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps

from uuid import uuid4, UUID
from PIL import Image
import cv2

from paddleocr import PaddleOCRVL


In [ ]:
settings = {
    "use_ocr_for_image_block": True,
    "use_doc_orientation_classify": True,
    "use_doc_unwarping": False,
    "use_chart_recognition": True,
    "use_layout_detection": True,
    "use_seal_recognition": True,
    "format_block_content": True,
    "merge_layout_blocks": True,
    "markdown_ignore_labels": [],
}

pipeline = PaddleOCRVL(
    vl_rec_api_model_name="PaddlePaddle/PaddleOCR-VL-1.5",
    **settings,
)

In [ ]:
def parse_document(document_path: str):
    document_path = Path(document_path)

    if not document_path.exists():
        raise FileNotFoundError(f"No file found at {document_path}")

    output = pipeline.predict(
        input=str(document_path),
        **settings,
    )

    if document_path.suffix.lower() == ".pdf":
        output = pipeline.restructure_pages(list(output))

    for res in output:
        res.save_to_json(save_path="output")
        res.save_to_markdown(save_path="output") 
    
    return output

    
file_path = "./data/sales_volume.png"
output = parse_document(file_path)

In [ ]:
import base64
from io import BytesIO


def crop_region(image, bbox, padding=0):
    """Crop a region from image with optional padding."""
    x1, y1, x2, y2 = bbox
    if not hasattr(image, 'width'):
        from PIL import Image
        image = Image.fromarray(image)

    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(image.width, x2 + padding)
    y2 = min(image.height, y2 + padding)
    return image.crop((x1, y1, x2, y2))


def image_to_base64(img):
    """Convert PIL Image to base64 string."""
    buffer = BytesIO()
    img.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

In [ ]:
from pydantic import BaseModel
from typing import List, Any


class Grounding(BaseModel):
    id: int
    global_id: int
    page_index: int
    bbox: List[int]
    score: float

    
class Chunk(BaseModel):
    chunk_id: UUID
    chunk_type: str
    chunk_markdown: str
    grounding: Grounding


class Metadata(BaseModel):
    filename: str
    page_image: Any
    page_image_base64: str
    page_index: int
    page_count: int
    
    
    
class Document(BaseModel):
    doc_id: UUID
    markdown: str
    chunks: List[Chunk]
    metadata: Metadata

In [ ]:
import json


def load_json(file_path):
    """
    Loads a JSON file from the specified path.
    """
    with open(file_path, 'r') as f:
        return json.load(f)


def load_markdown(file_path):
    """
    Loads a markdown file from the specified path.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()


json_files = sorted(list(Path("output").glob(f"**/{Path(file_path).stem}*.json")))
md_files = sorted(list(Path("output").glob(f"**/{Path(file_path).stem}*.md")))


In [ ]:
documents: List[Document] = []

for idx in range(len(json_files)):
    json_data = load_json(json_files[idx])
    md_data = load_markdown(md_files[idx])
    chunks_data = json_data["parsing_res_list"]
    layout_results = json_data['layout_det_res']['boxes']
    output_img = output[idx]["doc_preprocessor_res"]["output_img"]
    
    chunks: List[Chunk] = []
    for data, layout_item in zip(chunks_data, layout_results):
        chunks.append(
            Chunk(
                chunk_id=uuid4(),
                chunk_type=data["block_label"],
                chunk_markdown=data["block_content"],
                grounding=Grounding(
                    id=data["block_id"],
                    global_id=data.get("global_block_id", 0),
                    page_index= 0 if json_data["page_index"] == None else json_data["page_index"],
                    bbox=data["block_bbox"],
                    score=layout_item["score"]
                )
            )
        )
    document = Document(
        doc_id=uuid4(),
        markdown=md_data,
        chunks=chunks,
        metadata=Metadata(
            filename=json_data["input_path"],
            page_image=output_img,
            page_image_base64=image_to_base64(Image.fromarray(output_img)),
            page_index= 0 if json_data["page_index"] == None else json_data["page_index"],
            page_count= 1 if json_data["page_count"] == None else json_data["page_count"]
        )
    )
    documents.append(document)
         

In [ ]:
def visualize_layout(document, min_confidence=0.5):
    img_plot = document.metadata.page_image.copy()
    
    # Get all unique labels
    labels = list(set(chunk.chunk_type for chunk in document.chunks))
    
    # Generate colors dynamically from colormap
    cmap = colormaps.get_cmap('tab20') 
    color_map = {}
    for i, label in enumerate(labels):
        rgba = cmap(i % 20)
        # Convert to BGR (0-255) for OpenCV
        color_map[label] = (int(rgba[2]*255), int(rgba[1]*255), int(rgba[0]*255))
    
    for chunk in document.chunks:
        if chunk.grounding.score < min_confidence:
            continue
            
        label = chunk.chunk_type
        score = chunk.grounding.score
        coords = chunk.grounding.bbox
        
        color = color_map[label]
        
        x1, y1, x2, y2 = [int(c) for c in coords]
        pts = np.array([[x1, y1], [x2, y1], [x2, y2], [x1, y2]], dtype=int)
        
        cv2.polylines(img_plot, [pts], True, color, 2)
        text = f"{label} ({score:.2f})"
        cv2.putText(img_plot, text, (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    return img_plot

In [ ]:
def display_layout(document, min_confidence):
    """Display the layout detection results."""
    result_image = visualize_layout(document, min_confidence)

    plt.figure(figsize=(12, 14))
    plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Layout Detection Results")
    plt.show()


In [ ]:
display_layout(documents[0], min_confidence=0.0)

In [ ]:
crop_region(documents[0].metadata.page_image, bbox=documents[0].chunks[0].grounding.bbox)

In [ ]:
from IPython.display import display, Markdown, HTML

display(Markdown(documents[0].markdown))